# Regularized Regression

### Setup

In [ ]:
%pip install statsmodels

In [1]:
import statsmodels.api as sm
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
# load and inspect dataset (cleaned from project 1)
df = pd.read_csv("../data/auto_mpg.csv")

print(df.head())
print("\nShape:", df.shape)
print("\nColumns:", df.columns.tolist())

   cylinders  displacement  horsepower  weight  acceleration  model_year  \
0          8         307.0       130.0  3504.0          12.0          70   
1          8         350.0       165.0  3693.0          11.5          70   
2          8         318.0       150.0  3436.0          11.0          70   
3          8         304.0       150.0  3433.0          12.0          70   
4          8         302.0       140.0  3449.0          10.5          70   

   origin   mpg  
0       1  18.0  
1       1  15.0  
2       1  18.0  
3       1  16.0  
4       1  17.0  

Shape: (392, 8)

Columns: ['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year', 'origin', 'mpg']


In [3]:
# check data types and missing values
print(df.dtypes)
print(df.isnull().sum())

cylinders         int64
displacement    float64
horsepower      float64
weight          float64
acceleration    float64
model_year        int64
origin            int64
mpg             float64
dtype: object
cylinders       0
displacement    0
horsepower      0
weight          0
acceleration    0
model_year      0
origin          0
mpg             0
dtype: int64


In [4]:
# separate predictors and target
X = df.drop(columns=["mpg"])
y = df["mpg"]
print(X.head())

   cylinders  displacement  horsepower  weight  acceleration  model_year  \
0          8         307.0       130.0  3504.0          12.0          70   
1          8         350.0       165.0  3693.0          11.5          70   
2          8         318.0       150.0  3436.0          11.0          70   
3          8         304.0       150.0  3433.0          12.0          70   
4          8         302.0       140.0  3449.0          10.5          70   

   origin  
0       1  
1       1  
2       1  
3       1  
4       1  


In [5]:
# split data into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [6]:
# standardize predictors
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# add intercept
X_train_scaled = sm.add_constant(X_train_scaled)
X_test_scaled = sm.add_constant(X_test_scaled)

### Ridge Regression

In [7]:
# tune ridge alpha using 5-fold cross-validation and average RMSE

# alpha values to test
alphas = [0.001, 0.01, 0.1, 1, 10, 100]

# create 5 folds
kf = KFold(n_splits=5, shuffle=True, random_state=42)

ridge_cv_scores = []

# test every alpha 
for alpha in alphas:
    fold_rmse = []

    # go through all 5 folds
    for train_idx, val_idx in kf.split(X_train):
        
        # split current fold
        X_fold_train = X_train.iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]
        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]

        # standardize using fold training data only
        fold_scaler = StandardScaler()
        X_fold_train_scaled = fold_scaler.fit_transform(X_fold_train)
        X_fold_val_scaled = fold_scaler.transform(X_fold_val)

        # add intercept
        X_fold_train_scaled = sm.add_constant(X_fold_train_scaled)
        X_fold_val_scaled = sm.add_constant(X_fold_val_scaled)

        # set penalty weights
        alpha_weights = np.repeat(alpha, X_fold_train_scaled.shape[1])
        alpha_weights[0] = 0 

        # train ridge model
        ridge_cv_model = sm.OLS(
            y_fold_train,
            X_fold_train_scaled
        ).fit_regularized(
            alpha=alpha_weights,
            L1_wt=0
        )

        # predict validation fold
        y_val_pred = ridge_cv_model.predict(X_fold_val_scaled)

        # calculate validation RMSE
        rmse = np.sqrt(
            mean_squared_error(y_fold_val, y_val_pred)
        )

        fold_rmse.append(rmse)

    # average RMSE across 5 folds
    avg_rmse = np.mean(fold_rmse)
    ridge_cv_scores.append(avg_rmse)

    print(f"Alpha: {alpha}, Average RMSE: {avg_rmse:.4f}")

# select alpha with the lowest average RMSE
best_ridge_alpha = alphas[np.argmin(ridge_cv_scores)]
print("\nBest Ridge Alpha:", best_ridge_alpha)

Alpha: 0.001, Average RMSE: 3.3532
Alpha: 0.01, Average RMSE: 3.3490
Alpha: 0.1, Average RMSE: 3.4184
Alpha: 1, Average RMSE: 3.8973
Alpha: 10, Average RMSE: 6.1356
Alpha: 100, Average RMSE: 7.6559

Best Ridge Alpha: 0.01


In [8]:
# create penalty weights using best ridge alpha
ridge_alpha_weights = np.repeat(
    best_ridge_alpha,
    X_train_scaled.shape[1]
)
ridge_alpha_weights[0] = 0  # exclude intercept from penalty

In [9]:
# train ridge regression model
ridge_model = sm.OLS(
    y_train,
    X_train_scaled
).fit_regularized(
    alpha=ridge_alpha_weights,
    L1_wt=0 # ridge
)

In [10]:
# display ridge coefficients for each predictor

feature_names = ["const"] + X.columns.tolist() # include intercept with predictor names

# pair each coefficient with its feature name
ridge_coefficients = pd.Series(
    ridge_model.params,
    index=feature_names
)

print(ridge_coefficients)

const           23.599361
cylinders       -0.469692
displacement     0.977049
horsepower      -0.908489
weight          -4.667132
acceleration     0.006313
model_year       2.722611
origin           1.268601
dtype: float64


In [11]:
# predict and display actual vs. predicted MPG

y_pred_ridge = ridge_model.predict(X_test_scaled)

# compare actual MPG values with the model's predicted MPG values
ridge_results = pd.DataFrame({
    "Actual MPG": y_test.to_numpy(), # convert y_test from pandas series to numpy array
    "Predicted MPG": y_pred_ridge
})

print(ridge_results.head())

   Actual MPG  Predicted MPG
0        26.0      25.890771
1        21.6      26.167406
2        36.1      34.297227
3        26.0      24.800514
4        27.0      28.494060


In [12]:
# evaluate ridge model performance using RMSE and R^2

# calculate RMSE
ridge_rmse = np.sqrt(
    mean_squared_error(y_test, y_pred_ridge)
)

# calculate R^2
ridge_r2 = r2_score(
    y_test,
    y_pred_ridge
)

print("Ridge RMSE:", ridge_rmse)
print("Ridge R²:", ridge_r2)

Ridge RMSE: 3.2973096237408797
Ridge R²: 0.7869881168144173


### Lasso Regression

In [13]:
# tune lasso alpha using 5-fold cross-validation and average RMSE

# alpha values to test
alphas = [0.001, 0.01, 0.1, 1, 10, 100]

# create 5 folds
kf = KFold(n_splits=5, shuffle=True, random_state=42)

lasso_cv_scores = []

# test every alpha
for alpha in alphas:
    fold_rmse = []

    # go through all 5 folds
    for train_idx, val_idx in kf.split(X_train):

        # split current fold
        X_fold_train = X_train.iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]
        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]

        # standardize using fold training data only
        fold_scaler = StandardScaler()
        X_fold_train_scaled = fold_scaler.fit_transform(X_fold_train)
        X_fold_val_scaled = fold_scaler.transform(X_fold_val)

        # add intercept
        X_fold_train_scaled = sm.add_constant(X_fold_train_scaled)
        X_fold_val_scaled = sm.add_constant(X_fold_val_scaled)

        # set penalty weights
        alpha_weights = np.repeat(alpha, X_fold_train_scaled.shape[1])
        alpha_weights[0] = 0

        # train Lasso model
        lasso_cv_model = sm.OLS(
            y_fold_train,
            X_fold_train_scaled
        ).fit_regularized(
            alpha=alpha_weights,
            L1_wt=1
        )

        # predict validation fold
        y_val_pred = lasso_cv_model.predict(X_fold_val_scaled)

        # calculate validation RMSE
        rmse = np.sqrt(
            mean_squared_error(y_fold_val, y_val_pred)
        )

        fold_rmse.append(rmse)

    # average RMSE across 5 folds
    avg_rmse = np.mean(fold_rmse)
    lasso_cv_scores.append(avg_rmse)

    print(f"Alpha: {alpha}, Average RMSE: {avg_rmse:.4f}")

# select alpha with the lowest average RMSE
best_lasso_alpha = alphas[np.argmin(lasso_cv_scores)]

print("\nBest Lasso Alpha:", best_lasso_alpha)

Alpha: 0.001, Average RMSE: 3.3536
Alpha: 0.01, Average RMSE: 3.3437
Alpha: 0.1, Average RMSE: 3.3518
Alpha: 1, Average RMSE: 3.6104
Alpha: 10, Average RMSE: 7.9141
Alpha: 100, Average RMSE: 7.9141

Best Lasso Alpha: 0.01


In [14]:
# create penalty weights using best Lasso alpha
lasso_alpha_weights = np.repeat(
    best_lasso_alpha,
    X_train_scaled.shape[1]
)
lasso_alpha_weights[0] = 0  # exclude intercept from penalty

In [15]:
# train lasso regression model
lasso_model = sm.OLS(
    y_train,
    X_train_scaled
).fit_regularized(
    alpha=lasso_alpha_weights,
    L1_wt=1 # lasso
)

In [16]:
# display lasso coefficients for each predictor

# pair each coefficient with its feature name
lasso_coefficients = pd.Series(
    lasso_model.params.to_numpy(), # remove old labels and keep coefficient values
    index=feature_names
)

print(lasso_coefficients)

const           23.599361
cylinders       -0.382769
displacement     1.128755
horsepower      -0.839027
weight          -4.975586
acceleration     0.000000
model_year       2.764608
origin           1.258881
dtype: float64


In [17]:
# predict and display actual vs. predicted MPG

y_pred_lasso = lasso_model.predict(X_test_scaled)

# compare actual MPG values with predicted MPG values
lasso_results = pd.DataFrame({
    "Actual MPG": y_test.to_numpy(), # convert actual values to NumPy array
    "Predicted MPG": y_pred_lasso
})

print(lasso_results.head())

   Actual MPG  Predicted MPG
0        26.0      25.837619
1        21.6      26.086561
2        36.1      34.424332
3        26.0      24.822670
4        27.0      28.448806


In [18]:
# evaluate lasso model performance using RMSE and R^2

# calculate RMSE
lasso_rmse = np.sqrt(
    mean_squared_error(y_test, y_pred_lasso)
)

# calculate R^2
lasso_r2 = r2_score(
    y_test,
    y_pred_lasso
)

print("Lasso RMSE:", lasso_rmse)
print("Lasso R²:", lasso_r2)

Lasso RMSE: 3.2906920560669057
Lasso R²: 0.7878422713750637
